# Customer Churn Prediction Using Data Mining — ChurnIQ

**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`) — 7,043 customers, 21 attributes.

This notebook documents the full CRISP-DM style workflow: business understanding, data
understanding, data preparation, modelling, evaluation and deployment.

## 1. Setup and data loading

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG = ROOT / "docs" / "visualizations"
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Data understanding

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
df.describe(include="all").T.head(25)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("Churn distribution:")
print(df["Churn"].value_counts())
print()
print("Churn rate: {:.2%}".format((df["Churn"] == "Yes").mean()))

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.54%


### Missing / malformed values

`TotalCharges` is stored as text and contains 11 blank strings for brand-new customers.

In [5]:
blank = df["TotalCharges"].astype(str).str.strip() == ""
print("Blank TotalCharges rows:", int(blank.sum()))
df.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Blank TotalCharges rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


## 3. Data preparation (cleaning + preprocessing)

In [6]:
data = df.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["TotalCharges"] = data["TotalCharges"].fillna(data["tenure"] * data["MonthlyCharges"])
data = data.drop(columns=["customerID"])          # identifier: no predictive signal
data["Churn"] = (data["Churn"] == "Yes").astype(int)

print("Missing values after cleaning:", int(data.isna().sum().sum()))
print("Duplicate rows:", int(data.duplicated().sum()))
data.head()

Missing values after cleaning: 0
Duplicate rows: 22


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


### Exploratory visualisations

In [7]:
plt.style.use("seaborn-v0_8-darkgrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
data["Churn"].value_counts().rename({0: "Retained", 1: "Churned"}).plot.pie(
    autopct="%1.1f%%", colors=["#14b8a6", "#f97316"], ax=axes[0], ylabel="")
axes[0].set_title("Churn distribution")
(df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
   .plot.bar(color="#14b8a6", ax=axes[1]))
axes[1].set_title("Churn rate by contract type"); axes[1].set_ylabel("% churned")
plt.tight_layout(); plt.savefig(FIG / "eda_overview.png", dpi=130); plt.show()

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col in zip(axes, ["InternetService", "PaymentMethod", "TechSupport"]):
    (df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
       .plot.bar(ax=ax, color="#f97316"))
    ax.set_title(f"Churn rate by {col}"); ax.set_ylabel("% churned")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(FIG / "eda_categories.png", dpi=130); plt.show()

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col in zip(axes, ["tenure", "MonthlyCharges"]):
    ax.hist([data.loc[data.Churn == 0, col], data.loc[data.Churn == 1, col]],
            bins=25, stacked=True, color=["#14b8a6", "#f97316"], label=["Retained", "Churned"])
    ax.set_title(f"{col} distribution"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "eda_numeric.png", dpi=130); plt.show()

In [10]:
corr = data[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]].corr()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="viridis")
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
fig.colorbar(im); ax.set_title("Correlation matrix")
plt.tight_layout(); plt.savefig(FIG / "correlation.png", dpi=130); plt.show()

### Preprocessing pipeline

* **Numeric** (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`) → `StandardScaler`
* **Categorical** (16 attributes) → `OneHotEncoder(handle_unknown="ignore")`

Everything is wrapped in a `ColumnTransformer` inside each model `Pipeline`, so the exact
same transformations are applied at training and at prediction time.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

X = data.drop(columns=["Churn"])
y = data["Churn"]

numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in X.columns if c not in numeric]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Encoded feature count:", preprocessor.fit(X_train).transform(X_train).shape[1])

Train: (5634, 19)  Test: (1409, 19)
Encoded feature count: 45


## 4. Modelling

Three classic data-mining classifiers, all with `class_weight="balanced"` to counter the 73/27 class imbalance.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=25,
                                            class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=8,
                                            class_weight="balanced", random_state=42, n_jobs=-1),
}

pipelines = {name: Pipeline([("prep", preprocessor), ("clf", clf)]) for name, clf in models.items()}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    print(f"{name}: trained")

Logistic Regression: trained
Decision Tree: trained


Random Forest: trained


## 5. Evaluation

In [13]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

rows = []
for name, pipe in pipelines.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })
results = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
results.round(4)

,model,accuracy,precision,recall,f1,roc_auc
0,Random Forest,0.7551,0.5256,0.7968,0.6334,0.8420
1,Decision Tree,0.7445,0.5122,0.7861,0.6203,0.8339
2,Logistic Regression,0.7381,0.5043,0.7834,0.6136,0.8416


In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1", n_jobs=-1)
    print(f"{name}: 5-fold CV F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

Logistic Regression: 5-fold CV F1 = 0.6258 (+/- 0.0091)


Decision Tree: 5-fold CV F1 = 0.6083 (+/- 0.0061)


Random Forest: 5-fold CV F1 = 0.6346 (+/- 0.0123)


In [15]:
best_name = results.iloc[0]["model"]
best = pipelines[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best.predict(X_test), target_names=["Retained", "Churned"]))

Best model: Random Forest
              precision    recall  f1-score   support

    Retained       0.91      0.74      0.82      1035
     Churned       0.53      0.80      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.76      0.77      1409



In [16]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

cm = confusion_matrix(y_test, best.predict(X_test))
axes[0].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
axes[0].set_xticks([0, 1], ["Pred retain", "Pred churn"])
axes[0].set_yticks([0, 1], ["True retain", "True churn"])
axes[0].set_title(f"Confusion matrix — {best_name}")

for name, pipe in pipelines.items():
    fpr, tpr, _ = roc_curve(y_test, pipe.predict_proba(X_test)[:, 1])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curves"); axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "evaluation.png", dpi=130); plt.show()

### Feature importance

In [17]:
feature_names = best.named_steps["prep"].get_feature_names_out()
feature_names = [f.split("__", 1)[-1] for f in feature_names]
importances = pd.Series(best.named_steps["clf"].feature_importances_, index=feature_names)
top = importances.sort_values(ascending=False).head(15)[::-1]

fig, ax = plt.subplots(figsize=(8, 5.5))
top.plot.barh(color="#14b8a6", ax=ax)
ax.set_title(f"Top 15 features — {best_name}")
plt.tight_layout(); plt.savefig(FIG / "feature_importance.png", dpi=130); plt.show()
top[::-1].round(4)

Contract_Month-to-month                0.1473
tenure                                 0.1240
TotalCharges                           0.0910
Contract_Two year                      0.0741
OnlineSecurity_No                      0.0681
MonthlyCharges                         0.0612
TechSupport_No                         0.0520
InternetService_Fiber optic            0.0513
PaymentMethod_Electronic check         0.0407
Contract_One year                      0.0198
InternetService_DSL                    0.0188
OnlineBackup_No                        0.0174
OnlineSecurity_Yes                     0.0141
StreamingMovies_No internet service    0.0122
DeviceProtection_No                    0.0118
dtype: float64

## 6. Deployment artefacts

The fitted pipelines are persisted with `joblib` and served by the FastAPI backend (`backend/app.py`) and by the ChurnIQ React dashboard.

In [18]:
import joblib
MODELS = ROOT / "models"; MODELS.mkdir(exist_ok=True)
for name, pipe in pipelines.items():
    joblib.dump(pipe, MODELS / f"{name.lower().replace(' ', '_')}.pkl")
joblib.dump(best, MODELS / "best_model.pkl")
results.to_csv(MODELS / "model_results.csv", index=False)
print("Saved:", sorted(p.name for p in MODELS.glob('*')))

Saved: ['best_model.pkl', 'decision_tree.pkl', 'feature_importance.csv', 'logistic_regression.pkl', 'model_results.csv', 'preprocessing_pipeline.pkl', 'random_forest.pkl']


In [19]:
sample = X_test.iloc[[0]]
print(sample.T)
print("\nChurn probability:", round(float(best.predict_proba(sample)[0, 1]), 4))

                                      437
gender                               Male
SeniorCitizen                           0
Partner                               Yes
Dependents                            Yes
tenure                                 72
PhoneService                          Yes
MultipleLines                         Yes
InternetService               Fiber optic
OnlineSecurity                        Yes
OnlineBackup                          Yes
DeviceProtection                      Yes
TechSupport                           Yes
StreamingTV                           Yes
StreamingMovies                       Yes
Contract                         Two year
PaperlessBilling                      Yes
PaymentMethod     Credit card (automatic)
MonthlyCharges                     114.05
TotalCharges                       8468.2

Churn probability: 0.0345


## 7. Conclusions

* Random Forest is the best performer on F1 and ROC-AUC and is used as the default model in ChurnIQ.
* Month-to-month contracts, low tenure, fiber-optic internet, electronic-check payment and the absence
  of tech support / online security are the strongest churn drivers.
* Recall is deliberately favoured over precision (balanced class weights): missing a churner costs more
  than contacting a happy customer.
* **Retention actions:** migrate month-to-month customers to annual contracts, bundle tech support and
  online security, and target the first 12 months of tenure with onboarding offers.